# 新步距盒需圖測試(消除離群值).ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import pandas as pd
import math
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
def calculate_step_length(x1, y1, x2, y2):
    return math.hypot(x2 - x1, y2 - y1)


In [ ]:
def step_x(x1, x2):
    return abs(x2 - x1)


In [ ]:
def process_file(uploaded_file):
    all_data = pd.DataFrame()

    for file_name in uploaded_file.keys():
        df = pd.read_csv(io.BytesIO(uploaded_file[file_name]))
        all_data = pd.concat([all_data, df], ignore_index=True)


    kpts_columns = all_data['kpts'].str.split(',', expand=True)


    for i in range(kpts_columns.shape[1]):
        kpts_columns.rename(columns={i: f'kpt_{i+1}'}, inplace=True)


    all_data = all_data.join(kpts_columns, rsuffix='_kpts')


    if 'kpt_46' not in all_data.columns or 'kpt_49' not in all_data.columns:
        print(f"文件中缺少必要的關節點數據，無法計算步距")
        return [], float('nan')


    x16 = all_data['kpt_46'].astype(float)
    x17 = all_data['kpt_49'].astype(float)


    valid_rows = ~(x16.isna() | x17.isna())
    x16_valid = x16[valid_rows]
    x17_valid = x17[valid_rows]


    if len(x16_valid) == 0 or len(x17_valid) == 0:
        print(f"步距數據缺失，無法計算")
        return [], float('nan')


    step_lengths = [step_x(x16_valid.iloc[i], x17_valid.iloc[i]) for i in range(len(x16_valid))]


    if len(step_lengths) > 0:
        average_step_length = sum(step_lengths) / len(step_lengths)
    else:
        average_step_length = float('nan')

    return step_lengths, average_step_length


In [ ]:
category_results = {}


In [ ]:
for category in range(1, 5):
    print(f"請上傳第 {category} 類的CSV文件")
    uploaded_files = files.upload()


    step_lengths, avg_step_length = process_file(uploaded_files)


    q1 = pd.Series(step_lengths).quantile(0.25)
    q2 = pd.Series(step_lengths).quantile(0.5)
    q3 = pd.Series(step_lengths).quantile(0.75)


    category_results[category] = {
        'q1': [q1],
        'q2': [q2],
        'q3': [q3],
        'step_lengths': step_lengths,
        'average': avg_step_length
    }


In [ ]:
for category in range(1, 5):
    if category in category_results:
        avg_step_length = category_results[category]['average']
        print(f"Category {category} 平均步距: {avg_step_length}")
    else:
        print(f"Category {category} 無法計算平均步距")


In [ ]:
import numpy as np


In [ ]:
plt.figure(figsize=(10, 6))


data = [category_results[cat]['step_lengths'] for cat in range(1, 5)]


flat_data = [np.hstack(data[cat - 1]) for cat in range(1, 5)]


plt.boxplot(flat_data, patch_artist=True, boxprops=dict(facecolor='skyblue'), showfliers=True)


plt.title('Comparison of Step Lengths for All Categories', fontsize=16)
plt.xlabel('Category', fontsize=14)
plt.ylabel('Step Length', fontsize=14)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 18))


In [ ]:
plt.subplot(3, 1, 1)
q1_values = [category_results[category]['q1'][0] for category in range(1, 5)]
plt.plot([1, 2, 3, 4], q1_values, marker='o', color='blue', label='Q1')
plt.title('Q1 Step Length Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q1 Step Length', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.subplot(3, 1, 2)
q2_values = [category_results[category]['q2'][0] for category in range(1, 5)]
plt.plot([1, 2, 3, 4], q2_values, marker='o', color='blue', label='Q2')
plt.title('Q2 Step Length Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q2 Step Length', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.subplot(3, 1, 3)
q3_values = [category_results[category]['q3'][0] for category in range(1, 5)]
plt.plot([1, 2, 3, 4], q3_values, marker='o', color='blue', label='Q3')
plt.title('Q3 Step Length Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q3 Step Length', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.tight_layout()
plt.show()
